In [1]:
from datasets import load_dataset, load_from_disk
from replay.metrics import Recall, Precision, HitRate
import polars as pl
import faiss
import pickle
import scipy
import os
import pandas as pd
from tqdm import tqdm
import numpy as np
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [4]:
hist_len = 400

item_embeddings = np.load(f"data/item_embs_hist_len{hist_len}.npy")

with open(f'data/item2id_hist_len{hist_len}.pickle', 'rb') as f:
    item2id = pickle.load(f)

In [5]:
item_embeddings.shape

(4031902, 384)

In [7]:
item_embeddings = np.load("data/item_embs.npy")
item_embeddings.shape

(3613746, 384)

In [5]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .select(
        pl.col("user_id"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
    .filter(pl.col("rn") <= hist_len)
)

In [6]:
alpha = 0.9 #0.95

def calc_uservec(row):
    ids = [item2id[item["item_id"]] for item in row]
    user_item_embs = item_embeddings[ids]
    weights = np.reshape(alpha ** np.arange(len(ids)), (len(ids), 1))
    user_vec = np.sum(user_item_embs * weights, axis=0)
    norm = np.linalg.norm(user_vec)
    return (user_vec / norm).tolist()

seq_data = (
    train_interactions
    .groupby("user_id")
    .agg(
        pl.struct(["item_id", "stime"])
        .sort_by("stime", descending=True)
        .alias("events"),
    )
    .with_columns(
        pl.col("events").apply(calc_uservec).alias("user_vec")
    )
    .select("user_id", "user_vec")
)

In [12]:
seq_data.head(5)

user_id,user_vec
i64,list[f64]
37884964,"[-0.085568, 0.053947, … 0.046195]"
17051216,"[-0.044183, 0.026419, … -0.066636]"
1242580,"[-0.068479, 0.033925, … -0.049728]"
15445528,"[-0.099172, 0.072478, … 0.024828]"
20319564,"[-0.056321, 0.090515, … -0.00154]"


In [7]:
user_ids = seq_data["user_id"].to_list()
user_vectors = seq_data["user_vec"].to_list()

In [8]:
#index = faiss.read_index("../item_index.faiss")
index = faiss.read_index("../item2item/item_index_tiny.faiss")

In [9]:
#with open("../item_ids", "rb") as fp:
#    item_ids = pickle.load(fp)
with open("../item2item/item_ids", "rb") as fp:
    item_ids = pickle.load(fp)

In [10]:
batch_size = 1024
user_recs = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(user_ids), batch_size)):
    cur_vectors = np.array(user_vectors[i:i+batch_size])
    cur_ids = user_ids[i:i+batch_size]
    _, idx = index.search(cur_vectors, k=300)
    recs = replace_func(idx)
    cur_recs = {cur_ids[i]: recs[i, :].tolist() for i in range(len(cur_ids))}
    user_recs = {**user_recs, **cur_recs}

100%|██████████| 204/204 [25:01<00:00,  7.36s/it]


In [11]:
with open(f"user2vec_recs_hist_size{hist_len}_alpha={alpha}", "wb") as fp:   #Pickling
    pickle.dump(user_recs, fp)

In [12]:
del user_vectors

In [13]:
recs = (
    seq_data
    .with_columns(
        pl.col("user_id").apply(lambda x: user_recs[x]).alias("recs")
    )
)

In [15]:
recs_stats = (
    recs
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.max("recs_count").alias("max_recs_count"),
    )
)

recs_stats

min_recs_count,mean_recs_count,max_recs_count
i64,f64,i64
300,300.0,300


In [16]:
recs.rename({"recs": "user2vec_recs"}).select("user_id", "user2vec_recs").write_parquet("user2vec_recs_v2.parquet")

In [47]:
#recs.select("user_id", "recs").write_parquet("user2vec_recs.parquet")

In [14]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .filter(pl.col("event_id") == "item_view")
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs,
        on="user_id",
        how="inner"
    )
)

In [15]:
TOP_K_VALUES = [10, 100, 300]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [16]:
metrics # alpha = 0.9 hist_len = 400

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.022332,0.055628,0.081173,0.017536,0.006082,0.003456,0.127853,0.271749,0.353454,14736.0,31321.0,40738.0


In [21]:
metrics # alpha = 0.95 hist_len = 200

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.021331,0.054578,0.080463,0.016057,0.005945,0.00344,0.117442,0.2644,0.346955,13536.0,30474.0,39989.0


In [27]:
metrics # alpha = 0.9 hist_len = 200

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.022332,0.055628,0.081173,0.017536,0.006082,0.003456,0.127853,0.271749,0.353454,14736.0,31321.0,40738.0


In [48]:
metrics # alpha = 0.85 hist_len = 200

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.022756,0.055686,0.080689,0.018295,0.006098,0.003419,0.134126,0.27548,0.353818,15459.0,31751.0,40780.0
